![image_1780822049411.png](./image_1780822049411.png "image_1780822049411.png")

![image_1780822068080.png](./image_1780822068080.png "image_1780822068080.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window

# Initialize Spark session
spark = SparkSession.builder.appName("OrdersProductsData").getOrCreate()

# Define dataset for orders
orders_data = [
    (10, 201, 10),
    (11, 202, 3),
    (12, 203, 7),
    (13, 204, 2),
    (14, 205, 1),
    (15, 201, 5),
    (16, 203, 3)
]

orders_columns = ["order_id", "product_id", "quantity"]

# Create orders DataFrame
orders_df = spark.createDataFrame(orders_data, orders_columns)

# Define dataset for products
products_data = [
    (201, "Widget A", "Hardware", 25.0),
    (202, "Widget B", "Hardware", 15.5),
    (203, "License X", "Software", 199.99),
    (204, "Cable C", "Accessories", 9.99),
    (205, "Manual", "Books", 35.0)
]

products_columns = ["product_id", "name", "category", "price"]

# Create products DataFrame
products_df = spark.createDataFrame(products_data, products_columns)

# Show both DataFrames
print("Orders DataFrame:")
orders_df.show()

print("Products DataFrame:")
products_df.show()


In [0]:
result_df = (
    orders_df.join(products_df, on="product_id")
    .withColumn(
        "total_revenue",
        f.sum(f.col("price") * f.col("quantity")).over(Window.partitionBy("category")),
    )
    .select(f.col("category"), f.col("total_revenue"))
    .distinct()
    .orderBy(f.col("total_revenue").desc())
)
display(result_df)